In [18]:
from tinycombinator import IC, execute
from tinycombinator.ast import move
from tinycombinator.helpers import Context

c = IC(lambda x: x)
f = c.copy()

move(IC(None), f)

f, c

(Nul, λa a)

In [3]:
from tinycombinator.helpers import BACKEND


BACKEND.set("py")


c = IC(lambda x: x)(lambda x: x)
print(c)
print(c.run())
print(c)

(
  λa a
  λb b)
λa a
(
  λa a
  λb b)


In [ ]:
from tinycombinator import Tag

In [20]:
len(Tag)

12

In [23]:
Tag._member_names_

['App',
 'Lam',
 'Sup',
 'Dup',
 'Dup2',
 'Null',
 'Var',
 'Freed',
 'Prim',
 'Mat',
 'Dat',
 'intermediate_var']

In [25]:
Tag.__members__.values()

dict_values([App, Lam, Sup, Dup, Dup2, Null, Var, Freed, Prim, Mat, Dat, intermediate_var])

In [4]:
with Context(DEBUG=1,print_tree=False,BACKEND="c", TIMEIT=1):

  print(c.run())


( λa a λb b)
LOAD: 5 nodes
1: HANDLE App 0x128000000 -> Lam 0x128000018
move Lam 0x128000048 0x128000048 -> 0x128000030
move Lam 0x128000030 0x128000030 -> 0x128000000
λa a

1 steps: 0.000106 seconds 0.009 Mips
Final result:
λa a
λa a


In [5]:
def adt(constor):
  print("creating adt", constor.__name__)
  variants = {k: v for k, v in constor.__dict__.items() if not k.startswith("__")}

  class ADT:
    __name__ = constor.__name__

    def __repr__(self):
      return f"{self.__class__.__name__}({vars(self)})"

  def make_variant(name, fields):
    if fields == []:
      return ADT()
    else:
      def __init__(self, *args, **kwargs):
        for f in fields:
          setattr(self, f, kwargs[f])
      return type(name, (ADT,), {"__init__": __init__})

  for name, fields in variants.items():
    setattr(ADT, name, make_variant(name, fields))

  return ADT


@adt
class List:
  cons = ["head", "tail"]
  nil = []


# usage
xs = List.cons(head=0, tail=List.nil)

xs


creating adt List


cons({'head': 0, 'tail': ADT({})})

In [6]:
from dataclasses import dataclass

In [7]:
from typing import Dict, List

In [11]:

@dataclass
class VariantType:
  type: "Adt"
  name: str

  def __call__(self, *args): return Variant(self, args)
  def __repr__(self): return f"{self.name}"


@dataclass
class Variant:
  type: VariantType
  args: List[IC]

  def __repr__(self): return f"{self.type.name} {{{', '.join(f'{v}' for v in self.args)}}}"

@dataclass
class Adt:

  variants: Dict[str, list]

  def __init__(self, **variants): self.variants = variants

  def __repr__(self): return f"Adt({self.variants})"
  
  def match(self, **patterns): return Match(self, **patterns)

  def __getattr__(self, name): return VariantType(self, name)


class Match:
  def __init__(self, adt : Adt, **cases):
    assert sorted(cases.keys()) == sorted(adt.variants.keys())
    self.adt = adt
    self.cases = cases


LIST = Adt(nil=[], cons=["head", "tail"])

LIST.cons(0, LIST.nil())


cons {0, nil {}}

In [17]:
LIST.match(
  nil = 22,
  cons = lambda head, tail: 44
)


cons {0, nil {}}

In [155]:

List = Adt(nil=[], cons=["head", "tail"])
List.nil()

nil {}

In [156]:
lambda s: List.match(
  s,
  {
    List.nil: 22,
    "nil": 22,
    "cons": lambda head, tail: 44
  }  
)

<function __main__.<lambda>(s)>